# Lab 02 — Advanced COMPLETE & Helper Functions

Structured outputs, error handling, token management, and text chunking.

| Item | Detail |
|---|---|
| **Functions** | Structured Outputs, `TRY_COMPLETE`, `COUNT_TOKENS`, `SPLIT_TEXT_RECURSIVE_CHARACTER` |
| **Exam Domain** | 2.0 Gen AI & LLM Functions |
| **You'll learn** | JSON mode, error-safe calls, token estimation, text splitting |


> **Note:** `TRY_COMPLETE` and `SPLIT_TEXT_RECURSIVE_CHARACTER` currently have no `AI_` prefixed equivalent and remain under the `SNOWFLAKE.CORTEX` namespace.

## Step 1 — Environment Setup

> **AI SQL Functions** — This lab uses the preferred `AI_` prefixed functions. 
Equivalent legacy functions: `AI_COMPLETE` (replaces `SNOWFLAKE.CORTEX.COMPLETE`), `AI_COUNT_TOKENS` (replaces `SNOWFLAKE.CORTEX.COUNT_TOKENS`).

In [ ]:
USE ROLE DS_ROLE;
USE WAREHOUSE DS_WH;

CREATE DATABASE IF NOT EXISTS GENAI_LEARNING;
CREATE SCHEMA IF NOT EXISTS GENAI_LEARNING.PUBLIC;
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

## Step 2 — Structured Outputs (JSON Mode)

Force the LLM to return valid JSON. Critical for production pipelines where downstream code parses the output.

In [ ]:
SELECT AI_COMPLETE(
    'claude-haiku-4-5',
    [
        {'role': 'system', 'content': 'Extract product info as JSON with keys: product_name, category, price_range, sentiment.'},
        {'role': 'user', 'content': 'The new UltraBook Pro laptop is amazing for developers. At $1,299 it is a great deal for a high-end machine.'}
    ],
    {'response_format': {'type': 'json'}}
):choices[0]:messages::STRING AS structured_response;

## Step 3 — TRY_COMPLETE (Error-Safe Completion)

`TRY_COMPLETE` returns NULL instead of raising an error when the model call fails.
Use it in batch processing where one bad row shouldn't fail the entire query.

> **Note:** `TRY_COMPLETE` has no `AI_` prefixed equivalent yet and remains under the `SNOWFLAKE.CORTEX` namespace.

In [ ]:
SELECT
    'valid_call' AS test_case,
    SNOWFLAKE.CORTEX.TRY_COMPLETE('claude-haiku-4-5', 'What is 2+2?') AS response
UNION ALL
SELECT
    'invalid_model',
    SNOWFLAKE.CORTEX.TRY_COMPLETE('nonexistent-model-xyz', 'Hello');

## Step 4 — COUNT_TOKENS

Estimate token count for a given model. Essential for cost estimation and staying within context windows.

In [ ]:
SELECT
    'short' AS input_type,
    AI_COUNT_TOKENS('ai_complete', 'claude-haiku-4-5', 'Hello world') AS tokens
UNION ALL
SELECT 'medium',
    AI_COUNT_TOKENS('ai_complete', 'claude-haiku-4-5', 'Snowflake is a cloud data platform that enables data storage, processing, and analytic solutions. It provides a unique architecture that separates compute from storage.')
UNION ALL
SELECT 'long',
    AI_COUNT_TOKENS('ai_complete', 'claude-haiku-4-5', REPEAT('Snowflake Cortex AI is powerful. ', 100));

## Step 5 — SPLIT_TEXT_RECURSIVE_CHARACTER

Split long text into chunks suitable for embedding or LLM context windows.
The recursive character splitter tries to split on paragraph → sentence → word boundaries.

> **Note:** `SPLIT_TEXT_RECURSIVE_CHARACTER` has no `AI_` prefixed equivalent yet and remains under the `SNOWFLAKE.CORTEX` namespace.

In [ ]:
SELECT
    chunk.INDEX AS chunk_number,
    chunk.VALUE::STRING AS chunk_text,
    LENGTH(chunk.VALUE::STRING) AS chunk_length
FROM TABLE(FLATTEN(
    SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(
        'Snowflake is a cloud-native data platform. It separates compute from storage for elastic scaling. Virtual warehouses provide dedicated compute resources that auto-suspend and auto-resume. Time Travel enables accessing historical data. Zero-copy cloning creates instant copies without duplicating storage. Snowflake supports structured, semi-structured, and unstructured data natively. Cortex AI brings LLM capabilities directly into SQL workflows. Users can classify, summarize, translate, and extract information using simple SQL function calls.',
        'claude-haiku-4-5',
        {'max_tokens': 50}
    )
)) AS chunk;

## Key Takeaways

- **Structured Outputs**: Use `response_format: {type: json}` to force valid JSON.
- **TRY_COMPLETE**: Returns NULL on failure — safe for batch processing.
- **COUNT_TOKENS**: Estimate tokens before calling LLMs — essential for cost control.
- **SPLIT_TEXT_RECURSIVE_CHARACTER**: Chunk long text for embeddings or context windows.
